In [8]:
# Note: market 2's data is dummy as I only got market 1
import pandas
import os
from datetime import datetime, date, timedelta
from copy import deepcopy

# Read both market's data into data frame
df_1 = pandas.read_csv(os.path.join('data', 'market_1.csv'))
df_2 = pandas.read_csv(os.path.join('data', 'market_2.csv'))

# Read Attachment 1 as setting
param_df = pandas.read_csv('Attachment.csv')
PARAM = {}
for name, value in zip(param_df['Name'], param_df['Values']):
    PARAM[name] = value

In [29]:
PARAM

{'Max charging rate': 2.0,
 'Max discharging rate': 2.0,
 'Max storage volume': 4.0,
 'Battery charging efficiency': 0.05,
 'Battery discharging efficiency': 0.05,
 'Lifetime (1)': 10.0,
 'Lifetime (2)': 5000.0,
 'Storage volume degradation rate': 0.001,
 'Capex ': 500000.0,
 'Fixed Operational Costs': 5000.0}

In [9]:
# Use price history to predict after having 4-week data
holiday = [date(2018, 1, 1)]  # Only 1 holiday from 2018-01-01 to 2018-03-01

price_history = []  # As we don't know the future, everything should only base on history price
# list of dict: {time, is_holiday, market_1_price, market_2_price}
# use date().weekday() to find weekday. Both bank holiday and weekday = 5 or 6 defined as holiday
price_dict = {'time': datetime(2018, 1, 1), 'is_holiday': False, 'market_1_price': 0, 'market_2_price': 0}

market_1_ind = 0
market_2_ind = 0
current_time = datetime(2018, 1, 1)
current_battery = [0, 0]  # assume we have 2 battery initially, so that we can go for arbitrage profit without prediction

while market_1_ind < len(df_1) and market_2_ind < len(df_2):
    current_dict = deepcopy(price_dict)
    # No hard code, make it suitiable for flexible time as long as the time is orderred
    market_1_time = datetime.strptime(df_1['time'][market_1_ind], '%Y-%m-%d %H:%M:%S')
    market_2_time = datetime.strptime(df_2['time'][market_2_ind], '%Y-%m-%d %H:%M:%S')
    current_time = min(market_1_time, market_2_time)

    if current_time >= market_1_time:
        # Read market 1, get price, then move to the next row
        market_1_price = df_1['price'][market_1_ind]
        market_1_ind += 1

    if current_time >= market_2_time:
        # Read market 2, get price, then move to the next row
        market_2_price = df_2['price'][market_2_ind]
        market_2_ind += 1

    current_dict['time'] = current_time
    current_dict['is_holiday'] = current_time.weekday() in [5, 6] or current_time.date() in holiday
    current_dict['market_1_price'] = market_1_price
    current_dict['market_2_price'] = market_2_price
    price_history.append(current_dict)

In [15]:
def moving_average(history_data: list, market: int, days=7)  -> float:
    sum_price = 0
    count = 0
    current_time = history_data[-1]['time']
    for item in reversed(history_data):
        if item['time'] > current_time - timedelta(days=days):
            sum_price += item[f'market_{market}_price']
            count += 1
        else:
            break
    return sum_price / count

moving_average(price_history, 1)

np.float64(85.72738095238093)